# Assignment in Python _ Tam Nguyen Bang 475308

Main improvements:
- uses relative paths instead of hard-coded absolute paths
- replaces repeated code with loops and vectors
- extracts repeated logic into functions
- collects hard-coded parameters into a configuration section
- saves outputs to project folders for reproducibility

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# ---------------------------
# Configuration
# ---------------------------

HERE = Path.cwd()
DATA_DIR = HERE / "Data"
OUTPUT_DIR = HERE / "Outputs"
FIGURES_DIR = OUTPUT_DIR / "figures"

EMPLOYMENT_FILE = DATA_DIR / "Eurostat_employment_isco.xlsx"
TASK_FILE = DATA_DIR / "onet_tasks.csv"

ISCO_SHEETS = [f"ISCO{i}" for i in range(1, 10)]
COUNTRIES = ["Belgium", "Poland", "Spain"]
TASK_COLUMNS = ["t_4A2a4", "t_4A2b2", "t_4A4a1"]
TASK_GROUP_NAME = "NRCA"

OUTPUT_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

In [ ]:
def check_input_files(*paths: Path) -> None:
    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "Missing required input file(s):\n- " + "\n- ".join(missing)
        )


def weighted_standardize(series: pd.Series, weights: pd.Series) -> pd.Series:
    valid = series.notna() & weights.notna()
    x = series[valid]
    w = weights[valid]

    if x.empty:
        warnings.warn(f"No valid values found for {series.name}; returning NaNs.")
        return pd.Series(np.nan, index=series.index)

    if (w <= 0).any():
        warnings.warn(
            f"Non-positive weights found for {weights.name}; returning NaNs."
        )
        return pd.Series(np.nan, index=series.index)

    mu = np.average(x, weights=w)
    sd = np.sqrt(np.average((x - mu) ** 2, weights=w))

    if sd == 0:
        warnings.warn(f"Zero weighted standard deviation for {series.name}.")
        return pd.Series(np.nan, index=series.index)

    result = pd.Series(np.nan, index=series.index)
    result.loc[valid] = (x - mu) / sd
    return result


def load_employment_data(file_path: Path, sheet_names: list[str]) -> pd.DataFrame:
    sheets = []

    for idx, sheet_name in enumerate(sheet_names, start=1):
        sheet_df = pd.read_excel(file_path, sheet_name=sheet_name)
        sheet_df["ISCO"] = idx
        sheets.append(sheet_df)

    return pd.concat(sheets, ignore_index=True)


def add_country_totals_and_shares(df: pd.DataFrame, countries: list[str]) -> pd.DataFrame:
    out = df.copy()

    for country in countries:
        total_col = f"total_{country}"
        share_col = f"share_{country}"

        if country not in out.columns:
            warnings.warn(f"Country column '{country}' not found in employment data.")
            continue

        out[total_col] = out.groupby("TIME")[country].transform("sum")
        out[share_col] = out[country] / out[total_col]

    return out


def prepare_task_data(file_path: Path) -> pd.DataFrame:
    task_df = pd.read_csv(file_path).copy()
    task_df["isco08_1dig"] = task_df["isco08"].astype(str).str[:1].astype(int)

    agg = (
        task_df.groupby("isco08_1dig", as_index=False)
        .mean(numeric_only=True)
        .drop(columns=["isco08"], errors="ignore")
    )
    return agg


def standardize_task_columns(
    df: pd.DataFrame,
    countries: list[str],
    task_columns: list[str],
) -> pd.DataFrame:
    out = df.copy()

    for country in countries:
        weight_col = f"share_{country}"

        if weight_col not in out.columns:
            warnings.warn(f"Weight column '{weight_col}' not found; skipping {country}.")
            continue

        for task_col in task_columns:
            std_col = f"std_{country}_{task_col}"
            out[std_col] = weighted_standardize(out[task_col], out[weight_col])

    return out


def build_task_group_index(
    df: pd.DataFrame,
    countries: list[str],
    task_columns: list[str],
    group_name: str,
) -> pd.DataFrame:
    out = df.copy()
    aggregate_tables = {}

    for country in countries:
        std_task_cols = [f"std_{country}_{task_col}" for task_col in task_columns]
        weight_col = f"share_{country}"
        group_col = f"{country}_{group_name}"
        std_group_col = f"std_{country}_{group_name}"
        weighted_group_col = f"multip_{country}_{group_name}"

        missing_cols = [col for col in std_task_cols + [weight_col] if col not in out.columns]
        if missing_cols:
            warnings.warn(
                f"Skipping {country} because required columns are missing: {missing_cols}"
            )
            continue

        out[group_col] = out[std_task_cols].sum(axis=1)
        out[std_group_col] = weighted_standardize(out[group_col], out[weight_col])
        out[weighted_group_col] = out[std_group_col] * out[weight_col]

        aggregate_tables[country] = (
            out.groupby("TIME", as_index=False)[weighted_group_col]
            .sum()
            .rename(columns={weighted_group_col: group_col})
        )

    return out, aggregate_tables


def save_aggregates(aggregate_tables: dict[str, pd.DataFrame], output_dir: Path) -> None:
    combined_output = None

    for country, agg_df in aggregate_tables.items():
        file_path = output_dir / f"{country.lower()}_{TASK_GROUP_NAME.lower()}_timeseries.csv"
        agg_df.to_csv(file_path, index=False)

        if combined_output is None:
            combined_output = agg_df.copy()
        else:
            combined_output = combined_output.merge(agg_df, on="TIME", how="outer")

    if combined_output is not None:
        combined_output.to_csv(
            output_dir / f"all_countries_{TASK_GROUP_NAME.lower()}_timeseries.csv",
            index=False,
        )


def plot_aggregate_series(
    aggregate_tables: dict[str, pd.DataFrame],
    figures_dir: Path,
    group_name: str,
) -> None:
    for country, agg_df in aggregate_tables.items():
        y_col = f"{country}_{group_name}"

        plt.figure(figsize=(8, 4))
        plt.plot(agg_df["TIME"], agg_df[y_col])
        plt.xticks(range(0, len(agg_df), 3), agg_df["TIME"][::3], rotation=45)
        plt.title(f"{country} {group_name} over time")
        plt.tight_layout()
        plt.savefig(figures_dir / f"{country.lower()}_{group_name.lower()}.png", bbox_inches="tight")
        plt.close()

In [ ]:
# ---------------------------
# Run the analysis
# ---------------------------
check_input_files(TASK_FILE, EMPLOYMENT_FILE)

employment_data = load_employment_data(EMPLOYMENT_FILE, ISCO_SHEETS)
employment_data = add_country_totals_and_shares(employment_data, COUNTRIES)

task_agg = prepare_task_data(TASK_FILE)
combined = employment_data.merge(
    task_agg,
    left_on="ISCO",
    right_on="isco08_1dig",
    how="left",
)

combined = standardize_task_columns(combined, COUNTRIES, TASK_COLUMNS)
combined, aggregate_tables = build_task_group_index(
    combined,
    COUNTRIES,
    TASK_COLUMNS,
    TASK_GROUP_NAME,
)

save_aggregates(aggregate_tables, OUTPUT_DIR)
plot_aggregate_series(aggregate_tables, FIGURES_DIR, TASK_GROUP_NAME)

combined.head()

In [ ]:
# Review the aggregated country-level output
aggregate_tables